# Study 06: Ensemble Evaluation\n**Goal:** Compare individual models vs. ensemble predictions across Dice, IoU, HD95, ASD, and NSD metrics.

## 1. Setup

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.notebook import tqdm
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

## 2. Load Models & Data

In [ ]:
from src.config import DEVICE; from src.models import create_model
from src.metrics import SegmentationMetrics
from src.data_loader import DatasetConfig, DataPathManager, create_2d_dataloaders
from src.preprocessing import PreprocessingTransform

# Load models
models = {}
prefix = Path.cwd().parent
for name, path, pretrained in [
    ('best_model', 'models/best_model.pth', True),
    ('member_0', 'models/upre/member_0.pth', True),
    ('member_1', 'models/upre/member_1.pth', True),
    ('member_2', 'models/upre/member_2.pth', True),
]:
    m = create_model('mobilenetv2_unet', in_channels=1, out_channels=1, pretrained=pretrained)
    ckpt = torch.load(prefix/path, map_location='cpu', weights_only=True)
    if isinstance(ckpt, dict) and 'model_state' in ckpt: ckpt = ckpt['model_state']
    m.load_state_dict(ckpt, strict=False)
    models[name] = m.to(DEVICE)
    print(f'{name}: loaded from {path}')

# Test data
path_manager = DataPathManager(); volume_index = path_manager.build_index()
splitter = __import__('src.data_loader', fromlist=['VolumeWiseSplitter']).VolumeWiseSplitter()
splits = splitter.load_splits(DatasetConfig.SPLITS_DIR)
transform = PreprocessingTransform((256, 256), -100, 400)
_, _, test_loader = create_2d_dataloaders(
    volume_index, splits['train'], splits['val'], splits['test'],
    batch_size=1, transform_train=transform, transform_val=transform)
print(f'Test batches: {len(test_loader)}')

## 3. Run Inference

In [ ]:
metrics_fn = SegmentationMetrics()
results = {name: {'dice':[], 'iou':[], 'hd95':[], 'asd':[], 'nsd':[]} for name in models}
results['ensemble'] = {'dice':[], 'iou':[], 'hd95':[], 'asd':[], 'nsd':[]}

for batch in tqdm(test_loader, desc='Evaluating'):
    img = batch['image'].to(DEVICE)
    mask = batch['mask'].cpu().numpy()
    preds = []
    for name, m in models.items():
        m.eval()
        with torch.no_grad():
            p = (torch.sigmoid(m(img)) > 0.5).cpu().numpy().astype(np.uint8)
        preds.append(p)
        mets = metrics_fn(mask, p)
        for k in results[name]: results[name][k].append(mets[k])
    ensemble_pred = (np.stack(preds).mean(0) > 0.5).astype(np.uint8)
    mets = metrics_fn(mask, ensemble_pred)
    for k in results['ensemble']: results['ensemble'][k].append(mets[k])

print('Inference complete')

## 4. Results Summary

In [ ]:
summary = {}
for name, metrics in results.items():
    summary[name] = {k: {'mean': float(np.mean(v)), 'std': float(np.std(v)),
                          'median': float(np.median(v)), 'q25': float(np.percentile(v,25)),
                          'q75': float(np.percentile(v,75))} for k,v in metrics.items()}

df = pd.DataFrame({name: {k: f'{v["mean"]:.4f}\u00b1{v["std"]:.4f}' for k,v in m.items()}
                   for name, m in summary.items()}).T
print('='*80)
print('ENSEMBLE EVALUATION RESULTS')
print('='*80)
print(df[['dice','iou','hd95','asd','nsd']].to_string())

## 5. Dice Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
names = list(results.keys())
dice_data = [results[n]['dice'] for n in names]
bp = axes[0].boxplot(dice_data, labels=names, patch_artist=True)
colors = plt.cm.Set2(np.linspace(0, 1, len(names)))
for patch, c in zip(bp['boxes'], colors): patch.set_facecolor(c)
axes[0].set_ylabel('Dice'); axes[0].set_title('Dice Distribution per Model'); axes[0].grid(True, axis='y')
for i, n in enumerate(names):
    axes[1].bar(i, np.mean(results[n]['dice']), color=colors[i], alpha=0.7,
               yerr=np.std(results[n]['dice']), capsize=5)
axes[1].set_xticks(range(len(names))); axes[1].set_xticklabels(names, rotation=45)
axes[1].set_ylabel('Mean Dice'); axes[1].set_title('Mean Dice with Std Dev')
plt.tight_layout()
plt.savefig(Path.cwd().parent/'figures'/'ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Per-Sample Analysis

In [ ]:
# Identify best/worst cases
all_dice = results['ensemble']['dice']
worst_idx = np.argsort(all_dice)[:5]
best_idx = np.argsort(all_dice)[-5:]
print('Worst 5 samples (ensemble dice):')
for i in worst_idx:
    scores = {n: results[n]['dice'][i] for n in names}
    print(f'  Sample {i}: {scores}')
print('\nBest 5 samples (ensemble dice):')
for i in best_idx:
    scores = {n: results[n]['dice'][i] for n in names}
    print(f'  Sample {i}: {scores}')

## 7. Save Results

In [ ]:
out = Path.cwd().parent / 'results' / 'ensemble_evaluation.json'
with open(out, 'w') as f: json.dump(summary, f, indent=2)
print(f'Saved to {out}')

# Per-sample dice
per_sample = {n: results[n]['dice'] for n in names}
out2 = Path.cwd().parent / 'results' / 'per_sample_dice.json'
with open(out2, 'w') as f: json.dump(per_sample, f, indent=2)
print(f'Saved per-sample to {out2}')